# Desserialização

Desserializar é pegar dados (que podem chegar de várias formas) e criar e popular uma nova instância do modelo.

Já vimos uma forma de fazer isso:

In [1]:
from pydantic import BaseModel, ValidationError

In [2]:
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

In [3]:
p = Person(first_name='Isaac', last_name='Newton', age=84)

Pode não parecer desserialização, mas na essência foi isso: saímos de dados passados como argumentos nomeados para uma instância inicializada.

Também vimos que o Pydantic faz validação no caminho.

O Pydantic consegue desserializar dados em outros dois formatos:
- um `dict` Python
- uma string JSON

Vamos começar pelo `dict`:

In [4]:
data = {
    "first_name": "Isaac",
    "last_name": "Newton",
    "age": 84
}

Tecnicamente, dá para desserializar um dicionário desempacotando-o:

In [5]:
print(f"Person(**data): {Person(**data)!r}")

Person(**data): Person(first_name='Isaac', last_name='Newton', age=84)


Mas isso não é recomendado. Funciona em modelos simples, mas quando começamos a usar modelos mais complexos (com composição),
podemos ter problemas.

Em vez disso, o Pydantic oferece um método específico para carregar dados de um `dict`: `model_validate()`.

O nome do método deixa claro que o Pydantic não está só carregando dados — está também **validando**.

In [6]:
p = Person.model_validate(data)
print(f"p: {p!r}")

p: Person(first_name='Isaac', last_name='Newton', age=84)


As mesmas exceções de validação que vimos antes são levantadas quando há problemas:

In [7]:
missing_data = {"last_name": "Newton"}

try:
    Person.model_validate(missing_data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Person
first_name
  Field required [type=missing, input_value={'last_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
age
  Field required [type=missing, input_value={'last_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


O segundo formato que o Pydantic desserializa é JSON. Isso é enormemente útil com APIs REST, em que requisições e respostas
normalmente são JSON.

Para isso, o Pydantic oferece o método `model_validate_json()`.

In [8]:
data_json = '''
{
    "first_name": "Isaac",
    "last_name": "Newton",
    "age": 84
}
'''

In [9]:
p = Person.model_validate_json(data_json)
print(f"p: {p!r}")

p: Person(first_name='Isaac', last_name='Newton', age=84)


E as exceções de validação funcionam do mesmo jeito:

In [10]:
missing_data_json = '''
{
    "last_name": "Newton"
}
'''

try:
    Person.model_validate_json(missing_data_json)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Person
first_name
  Field required [type=missing, input_value={'last_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
age
  Field required [type=missing, input_value={'last_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
